# 10 — Train VAE

Short training run using `poregen` library modules.  
Run on an interactive HPC node with GPU(s).

In [ ]:
import yaml
from pathlib import Path
from torch.utils.data import DataLoader
import torch

from poregen.dataset.loader import PatchDataset
from poregen.models.vae import build_vae
from poregen.losses import compute_total_loss
from poregen.training import (
    seed_everything,
    select_device,
    get_autocast_dtype,
    make_scaler,
    train_loop,
)

## Config

In [ ]:
CFG_PATH = Path("../src/poregen/configs/example_vae.yaml")
cfg = yaml.safe_load(CFG_PATH.read_text())
print(cfg)

## Setup

In [ ]:
seed_everything(cfg["training"]["seed"])
device = select_device()  # or select_device(gpu_id=1)
autocast_dtype = get_autocast_dtype(device)
scaler = make_scaler(device)
print(f"Device: {device}  |  AMP dtype: {autocast_dtype}")

## Data

In [ ]:
DATA_ROOT = Path("../data/processed")

train_ds = PatchDataset(DATA_ROOT / "patch_index.parquet", DATA_ROOT, split="train")
val_ds   = PatchDataset(DATA_ROOT / "patch_index.parquet", DATA_ROOT, split="val")

train_loader = DataLoader(
    train_ds,
    batch_size=cfg["training"]["batch_size"],
    shuffle=True,
    num_workers=cfg["training"]["num_workers"],
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg["training"]["batch_size"],
    shuffle=False,
    num_workers=cfg["training"]["num_workers"],
    pin_memory=True,
)

print(f"Train patches: {len(train_ds):,}  |  Val patches: {len(val_ds):,}")

## Model + Optimizer

In [ ]:
model = build_vae(
    cfg["model"]["name"],
    z_channels=cfg["model"]["z_channels"],
    base_channels=cfg["model"]["base_channels"],
    n_blocks=cfg["model"]["n_blocks"],
    patch_size=cfg["model"]["patch_size"],
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["training"]["lr"])
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

## Loss function

In [ ]:
loss_cfg = cfg["loss"]
loss_fn = lambda output, batch, step: compute_total_loss(output, batch, step, loss_cfg)

## Train

In [ ]:
EXP_NAME = "conv_baseline"  # change per experiment

history = train_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    scaler,
    loss_fn,
    total_steps=cfg["training"]["total_steps"],
    eval_every=cfg["training"]["eval_every"],
    save_every=cfg["training"]["save_every"],
    run_dir=f"../runs/vae/{EXP_NAME}",
    device=device,
    autocast_dtype=autocast_dtype,
)

print(f"Done. Final loss: {history[-1]['total']:.4f}")

## Quick loss plot

In [ ]:
import matplotlib.pyplot as plt

train_h = [r for r in history if r["split"] == "train"]
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot([r["step"] for r in train_h], [r["total"] for r in train_h], lw=0.6)
ax.set(xlabel="Step", ylabel="Total loss", title="Training loss")
plt.tight_layout()
plt.show()